# Notebook 00 — Setup: Run This First, Then Enjoy the Presentation

**Estimated runtime: ~15-20 minutes on XL warehouse**

This notebook creates:
- **2 Billion** transactions (JPMC consumer banking)
- Table variants for each performance demo:
  - `TRANSACTIONS` — naturally ordered by date (baseline)
  - `TRANSACTIONS_UNORDERED` — same data, randomly shuffled
  - `TRANSACTIONS_CLUSTERED` — organized by account_id
  - `TRANSACTIONS_SOS` — with Search Optimization on account_id & transaction_id
  - `TRANSACTIONS_NO_SOS` — without Search Optimization (baseline)
- Supporting tables: `ACCOUNTS` (50M), `CUSTOMERS` (20M)

**Run all cells now and come back after the presentation.**

---
## Step 1: Infrastructure

In [ ]:
USE ROLE SYSADMIN;

CREATE DATABASE IF NOT EXISTS JPMC_PERF_HOL_V2;
CREATE SCHEMA IF NOT EXISTS JPMC_PERF_HOL_V2.CONSUMER_BANKING;
USE SCHEMA JPMC_PERF_HOL_V2.CONSUMER_BANKING;

CREATE WAREHOUSE IF NOT EXISTS HOL_XS WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
CREATE WAREHOUSE IF NOT EXISTS HOL_M  WAREHOUSE_SIZE = 'MEDIUM'  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
CREATE WAREHOUSE IF NOT EXISTS HOL_XL WAREHOUSE_SIZE = 'XLARGE'  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

---
## Step 2: Generate 2B Transactions

Data is first loaded into `TRANSACTIONS_TEMP` (4 batches of 500M), then sorted by `transaction_date` into `TRANSACTIONS`. This ensures natural date-ordering for partition pruning.

In [ ]:
USE WAREHOUSE HOL_XL;

CREATE OR REPLACE TABLE TRANSACTIONS_TEMP (
    transaction_id    STRING,
    account_id        STRING,
    transaction_date  TIMESTAMP_NTZ,
    merchant_name     STRING,
    amount            NUMBER(12,2),
    channel           STRING,
    transaction_type  STRING,
    status            STRING,
    fraud_flag        BOOLEAN,
    category          STRING,
    branch_id         STRING,
    fee_amount        NUMBER(8,2),
    balance_after     NUMBER(14,2)
);

In [ ]:
-- Batch 1 of 4: Jan 2023 - Jun 2023
INSERT INTO TRANSACTIONS_TEMP
SELECT
    'TXN-' || LPAD(SEQ8()::STRING, 12, '0')                              AS transaction_id,
    'ACC-' || LPAD(UNIFORM(0, 49999999, RANDOM())::STRING, 10, '0')       AS account_id,
    DATEADD(second, UNIFORM(0, 15551999, RANDOM()), '2023-01-01'::TIMESTAMP_NTZ) AS transaction_date,
    ARRAY_CONSTRUCT('Amazon','Walmart','Starbucks','Target','Costco','Shell','Uber','Netflix','Apple','Chase ATM',
                    'Whole Foods','Home Depot','Walgreens','McDonalds','Delta Airlines','Marriott','Verizon',
                    'CVS Pharmacy','Grubhub','Spotify')[UNIFORM(0,19,RANDOM())]::STRING AS merchant_name,
    ROUND(UNIFORM(1, 50000, RANDOM()) / 100.0, 2)                         AS amount,
    ARRAY_CONSTRUCT('mobile','web','branch','ATM','POS')[UNIFORM(0,4,RANDOM())]::STRING AS channel,
    ARRAY_CONSTRUCT('purchase','payment','transfer','withdrawal','deposit','refund')[UNIFORM(0,5,RANDOM())]::STRING AS transaction_type,
    ARRAY_CONSTRUCT('completed','pending','declined','reversed')[UNIFORM(0,3,RANDOM())]::STRING AS status,
    IFF(UNIFORM(0, 1000, RANDOM()) < 3, TRUE, FALSE)                      AS fraud_flag,
    ARRAY_CONSTRUCT('groceries','dining','travel','entertainment','utilities','healthcare',
                    'shopping','fuel','subscriptions','cash')[UNIFORM(0,9,RANDOM())]::STRING AS category,
    'BR-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 4, '0')             AS branch_id,
    ROUND(IFF(UNIFORM(0,100,RANDOM()) < 10, UNIFORM(1,500,RANDOM())/100.0, 0), 2) AS fee_amount,
    ROUND(UNIFORM(100, 5000000, RANDOM()) / 100.0, 2)                     AS balance_after
FROM TABLE(GENERATOR(ROWCOUNT => 500000000));

In [ ]:
-- Batch 2 of 4: Jul 2023 - Dec 2023
INSERT INTO TRANSACTIONS_TEMP
SELECT
    'TXN-' || LPAD((500000000 + SEQ8())::STRING, 12, '0')                 AS transaction_id,
    'ACC-' || LPAD(UNIFORM(0, 49999999, RANDOM())::STRING, 10, '0')       AS account_id,
    DATEADD(second, UNIFORM(0, 15897599, RANDOM()), '2023-07-01'::TIMESTAMP_NTZ) AS transaction_date,
    ARRAY_CONSTRUCT('Amazon','Walmart','Starbucks','Target','Costco','Shell','Uber','Netflix','Apple','Chase ATM',
                    'Whole Foods','Home Depot','Walgreens','McDonalds','Delta Airlines','Marriott','Verizon',
                    'CVS Pharmacy','Grubhub','Spotify')[UNIFORM(0,19,RANDOM())]::STRING AS merchant_name,
    ROUND(UNIFORM(1, 50000, RANDOM()) / 100.0, 2)                         AS amount,
    ARRAY_CONSTRUCT('mobile','web','branch','ATM','POS')[UNIFORM(0,4,RANDOM())]::STRING AS channel,
    ARRAY_CONSTRUCT('purchase','payment','transfer','withdrawal','deposit','refund')[UNIFORM(0,5,RANDOM())]::STRING AS transaction_type,
    ARRAY_CONSTRUCT('completed','pending','declined','reversed')[UNIFORM(0,3,RANDOM())]::STRING AS status,
    IFF(UNIFORM(0, 1000, RANDOM()) < 3, TRUE, FALSE)                      AS fraud_flag,
    ARRAY_CONSTRUCT('groceries','dining','travel','entertainment','utilities','healthcare',
                    'shopping','fuel','subscriptions','cash')[UNIFORM(0,9,RANDOM())]::STRING AS category,
    'BR-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 4, '0')             AS branch_id,
    ROUND(IFF(UNIFORM(0,100,RANDOM()) < 10, UNIFORM(1,500,RANDOM())/100.0, 0), 2) AS fee_amount,
    ROUND(UNIFORM(100, 5000000, RANDOM()) / 100.0, 2)                     AS balance_after
FROM TABLE(GENERATOR(ROWCOUNT => 500000000));

In [ ]:
-- Batch 3 of 4: Jan 2024 - Jun 2024
INSERT INTO TRANSACTIONS_TEMP
SELECT
    'TXN-' || LPAD((1000000000 + SEQ8())::STRING, 12, '0')                AS transaction_id,
    'ACC-' || LPAD(UNIFORM(0, 49999999, RANDOM())::STRING, 10, '0')       AS account_id,
    DATEADD(second, UNIFORM(0, 15638399, RANDOM()), '2024-01-01'::TIMESTAMP_NTZ) AS transaction_date,
    ARRAY_CONSTRUCT('Amazon','Walmart','Starbucks','Target','Costco','Shell','Uber','Netflix','Apple','Chase ATM',
                    'Whole Foods','Home Depot','Walgreens','McDonalds','Delta Airlines','Marriott','Verizon',
                    'CVS Pharmacy','Grubhub','Spotify')[UNIFORM(0,19,RANDOM())]::STRING AS merchant_name,
    ROUND(UNIFORM(1, 50000, RANDOM()) / 100.0, 2)                         AS amount,
    ARRAY_CONSTRUCT('mobile','web','branch','ATM','POS')[UNIFORM(0,4,RANDOM())]::STRING AS channel,
    ARRAY_CONSTRUCT('purchase','payment','transfer','withdrawal','deposit','refund')[UNIFORM(0,5,RANDOM())]::STRING AS transaction_type,
    ARRAY_CONSTRUCT('completed','pending','declined','reversed')[UNIFORM(0,3,RANDOM())]::STRING AS status,
    IFF(UNIFORM(0, 1000, RANDOM()) < 3, TRUE, FALSE)                      AS fraud_flag,
    ARRAY_CONSTRUCT('groceries','dining','travel','entertainment','utilities','healthcare',
                    'shopping','fuel','subscriptions','cash')[UNIFORM(0,9,RANDOM())]::STRING AS category,
    'BR-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 4, '0')             AS branch_id,
    ROUND(IFF(UNIFORM(0,100,RANDOM()) < 10, UNIFORM(1,500,RANDOM())/100.0, 0), 2) AS fee_amount,
    ROUND(UNIFORM(100, 5000000, RANDOM()) / 100.0, 2)                     AS balance_after
FROM TABLE(GENERATOR(ROWCOUNT => 500000000));

In [ ]:
-- Batch 4 of 4: Jul 2024 - Dec 2024
INSERT INTO TRANSACTIONS_TEMP
SELECT
    'TXN-' || LPAD((1500000000 + SEQ8())::STRING, 12, '0')                AS transaction_id,
    'ACC-' || LPAD(UNIFORM(0, 49999999, RANDOM())::STRING, 10, '0')       AS account_id,
    DATEADD(second, UNIFORM(0, 15897599, RANDOM()), '2024-07-01'::TIMESTAMP_NTZ) AS transaction_date,
    ARRAY_CONSTRUCT('Amazon','Walmart','Starbucks','Target','Costco','Shell','Uber','Netflix','Apple','Chase ATM',
                    'Whole Foods','Home Depot','Walgreens','McDonalds','Delta Airlines','Marriott','Verizon',
                    'CVS Pharmacy','Grubhub','Spotify')[UNIFORM(0,19,RANDOM())]::STRING AS merchant_name,
    ROUND(UNIFORM(1, 50000, RANDOM()) / 100.0, 2)                         AS amount,
    ARRAY_CONSTRUCT('mobile','web','branch','ATM','POS')[UNIFORM(0,4,RANDOM())]::STRING AS channel,
    ARRAY_CONSTRUCT('purchase','payment','transfer','withdrawal','deposit','refund')[UNIFORM(0,5,RANDOM())]::STRING AS transaction_type,
    ARRAY_CONSTRUCT('completed','pending','declined','reversed')[UNIFORM(0,3,RANDOM())]::STRING AS status,
    IFF(UNIFORM(0, 1000, RANDOM()) < 3, TRUE, FALSE)                      AS fraud_flag,
    ARRAY_CONSTRUCT('groceries','dining','travel','entertainment','utilities','healthcare',
                    'shopping','fuel','subscriptions','cash')[UNIFORM(0,9,RANDOM())]::STRING AS category,
    'BR-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 4, '0')             AS branch_id,
    ROUND(IFF(UNIFORM(0,100,RANDOM()) < 10, UNIFORM(1,500,RANDOM())/100.0, 0), 2) AS fee_amount,
    ROUND(UNIFORM(100, 5000000, RANDOM()) / 100.0, 2)                     AS balance_after
FROM TABLE(GENERATOR(ROWCOUNT => 500000000));

In [ ]:
-- Verify: should show 2,000,000,000 rows
SELECT COUNT(*) AS total_transactions FROM TRANSACTIONS_TEMP;

In [ ]:
-- Create TRANSACTIONS ordered by date (ensures natural partition pruning on date)
CREATE OR REPLACE TABLE TRANSACTIONS AS
SELECT * FROM TRANSACTIONS_TEMP
ORDER BY transaction_date;

-- Drop temp table
DROP TABLE TRANSACTIONS_TEMP;

---
## Step 3: Create Table Variants

In [ ]:
-- UNORDERED: Same 2B rows shuffled randomly (for Natural Ordering demo)
CREATE OR REPLACE TABLE TRANSACTIONS_UNORDERED AS
SELECT * FROM TRANSACTIONS
ORDER BY RANDOM();

In [ ]:
-- CLUSTERED: 6-column narrow table ordered by account_id (for Clustering demo)
CREATE OR REPLACE TABLE TRANSACTIONS_CLUSTERED AS
SELECT transaction_id, account_id, transaction_date, merchant_name, amount, channel
FROM TRANSACTIONS
ORDER BY account_id;

In [ ]:
-- SOS: Clone with Search Optimization (for SOS demo)
CREATE OR REPLACE TABLE TRANSACTIONS_SOS CLONE TRANSACTIONS;
ALTER TABLE TRANSACTIONS_SOS ADD SEARCH OPTIMIZATION ON EQUALITY(account_id, transaction_id);

In [ ]:
-- NO_SOS: Clone without SOS (baseline for SOS demo)
CREATE OR REPLACE TABLE TRANSACTIONS_NO_SOS CLONE TRANSACTIONS;

---
## Step 4: Supporting Tables

In [ ]:
-- ACCOUNTS: 50M rows
CREATE OR REPLACE TABLE ACCOUNTS AS
SELECT
    'ACC-' || LPAD(SEQ8()::STRING, 10, '0')                               AS account_id,
    'CUST-' || LPAD(UNIFORM(0, 19999999, RANDOM())::STRING, 10, '0')      AS customer_id,
    ARRAY_CONSTRUCT('checking','savings','credit_card','money_market','CD')[UNIFORM(0,4,RANDOM())]::STRING AS account_type,
    DATEADD(day, -UNIFORM(0, 3650, RANDOM()), '2024-12-31'::DATE)         AS open_date,
    'BR-' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 4, '0')             AS branch_id,
    ARRAY_CONSTRUCT('mass_market','affluent','high_net_worth','private_banking')[UNIFORM(0,3,RANDOM())]::STRING AS segment
FROM TABLE(GENERATOR(ROWCOUNT => 50000000));

In [ ]:
-- CUSTOMERS: 20M rows
CREATE OR REPLACE TABLE CUSTOMERS AS
SELECT
    'CUST-' || LPAD(SEQ8()::STRING, 10, '0')                              AS customer_id,
    'Customer_' || SEQ8()::STRING                                           AS customer_name,
    ARRAY_CONSTRUCT('mass_market','affluent','high_net_worth','private_banking')[UNIFORM(0,3,RANDOM())]::STRING AS segment,
    ARRAY_CONSTRUCT('NY','CA','TX','FL','IL','PA','OH','GA','NC','MI',
                    'NJ','VA','WA','AZ','MA','CO','TN','MD','MN','WI')[UNIFORM(0,19,RANDOM())]::STRING AS state,
    DATEADD(day, -UNIFORM(0, 5475, RANDOM()), '2024-12-31'::DATE)         AS signup_date
FROM TABLE(GENERATOR(ROWCOUNT => 20000000));

---
## Step 5: Verify Everything

In [ ]:
SELECT 'TRANSACTIONS' AS table_name, COUNT(*) AS row_count FROM TRANSACTIONS
UNION ALL SELECT 'TRANSACTIONS_UNORDERED', COUNT(*) FROM TRANSACTIONS_UNORDERED
UNION ALL SELECT 'TRANSACTIONS_CLUSTERED', COUNT(*) FROM TRANSACTIONS_CLUSTERED
UNION ALL SELECT 'TRANSACTIONS_SOS', COUNT(*) FROM TRANSACTIONS_SOS
UNION ALL SELECT 'TRANSACTIONS_NO_SOS', COUNT(*) FROM TRANSACTIONS_NO_SOS
UNION ALL SELECT 'ACCOUNTS', COUNT(*) FROM ACCOUNTS
UNION ALL SELECT 'CUSTOMERS', COUNT(*) FROM CUSTOMERS;

In [ ]:
-- Check SOS build progress (may still be building)
SHOW SEARCH OPTIMIZATION ON TRANSACTIONS_SOS;

In [ ]:
-- Suspend XL warehouse to save credits
ALTER WAREHOUSE HOL_XL SUSPEND;

---
## Setup Complete!

| Table | Rows | Purpose |
|-------|------|---------|
| TRANSACTIONS | 2B | Naturally ordered by date (date-range pruning) |
| TRANSACTIONS_UNORDERED | 2B | Shuffled (shows what happens without ordering) |
| TRANSACTIONS_CLUSTERED | 2B | Ordered by account_id (account lookup pruning) |
| TRANSACTIONS_SOS | 2B | Search Optimization on account_id & transaction_id |
| TRANSACTIONS_NO_SOS | 2B | Baseline without SOS |
| ACCOUNTS | 50M | Account metadata for JOINs |
| CUSTOMERS | 20M | Customer segmentation for JOINs |

**SOS Note**: Search Optimization builds asynchronously. If Notebook 03 shows no improvement, wait 10 more minutes and retry.

**Next** → Proceed to [Notebook 01 — Natural Ordering](./Notebook_01_Natural_Ordering.ipynb)